In [3]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
os.environ["JAX_ENABLE_X64"]="false"

import multiprocessing
os.environ["XLA_FLAGS"]="--xla_force_host_platform_device_count={}".format(multiprocessing.cpu_count())
# os.environ["XLA_FLAGS"]="--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=16"
import jax
# jax.config.update("jax_log_compiles", True)
# print(multiprocessing.cpu_count())
# print(jax.device_count())
jax.config.update("jax_enable_x64", False)


from sampling import *
from wavefunction import *
from observables import *
from training import *
from utils import *
from fit import *
from fitting_widget import *
import time


from plotly import graph_objects as go
from scipy.optimize import curve_fit

import gvar
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import kn

In [ ]:
file = ''
C=np.load(file, allow_pickle=True)['C']
C_cov = np.load(file, allow_pickle=True)['cov']

In [1]:
def xi_second_moment_from_corr(C, C_cov=None):
    """
    Compute the second-moment correlation length from the translation-averaged
    spatial correlator C(x) on a 1D periodic lattice:

        xi^2 = [1 / (4 sin^2(k_min / 2))] * ( Gtilde(0) / Gtilde(k_min) - 1 )

    with
        k_min = 2*pi / L
        Gtilde(k) = sum_x exp(i k x) G(x)

    Parameters
    ----------
    C : array-like, shape (L,)
        Real-space correlator C(x), typically from:
            C, C_cov, C_uncerts = Cr_with_cov_optimized(samples)
    C_cov : array-like, shape (L, L), optional
        Covariance matrix of C. If provided, an uncertainty on xi is returned
        using linear error propagation.

    Returns
    -------
    xi : float
        Second-moment correlation length.
    xi_err : float or None
        Uncertainty estimate from C_cov if provided, else None.
    """
    C = np.asarray(C, dtype=float)
    L = C.shape[0]
    k_min = 2.0 * np.pi / L

    x = np.arange(L, dtype=float)

    # Because C(x) is real and translationally invariant, Gtilde(k_min) should be real.
    G0 = np.sum(C)
    Gk = np.sum(np.exp(1j * k_min * x) * C).real

    pref = 1.0 / (4.0 * np.sin(k_min / 2.0) ** 2)
    xi2 = pref * (G0 / Gk - 1.0)

    # Numerical guard
    xi2 = max(xi2, 0.0)
    xi = np.sqrt(xi2)

    if C_cov is None:
        return xi, None

    C_cov = np.asarray(C_cov, dtype=float)

    # Linear propagation through xi(C)
    # xi^2 = pref * (G0/Gk - 1)
    # d(xi^2)/dC_j = pref * (1/Gk - G0/Gk^2 * dGk/dC_j)
    # where dGk/dC_j = cos(k*j), since xi^2 depends only on Re Gk
    cos_kx = np.cos(k_min * x)
    dxi2_dC = pref * (1.0 / Gk - (G0 / (Gk * Gk)) * cos_kx)

    if xi > 0.0:
        dxi_dC = 0.5 * dxi2_dC / xi
        xi_var = dxi_dC @ C_cov @ dxi_dC
        xi_err = np.sqrt(max(xi_var, 0.0))
    else:
        xi_err = 0.0

    return xi, xi_err

def corr_pbc_exp_over_sqrt(r, A, xi, L, eps=1e-12):
    """
    PBC-aware model with no constant term:

      C(r) = A * [ exp(-r/xi)/sqrt(r) + exp(-(L-r)/xi)/sqrt(L-r) ]

    Notes:
      - You should not include r=0 in the fit (singular 1/sqrt(r)).
      - eps is only a safety for sqrt when r is extremely small; prefer rmin>=1.
    """
    r = np.asarray(r, dtype=float)
    r1 = np.maximum(r, eps)
    r2 = np.maximum(L - r, eps)
    # return A * (np.exp(-r1 / xi) / np.sqrt(r1) + np.exp(-r2 / xi) / np.sqrt(r2))
    return A * (kn(0, r1/xi) + kn(0, r2/xi))



def fit_xi_windows_correlated(C, C_cov, L, rmin=1, rmax_stop=None, jitter=0.0):
    """
    Correlated (GLS) fits on growing windows [rmin..rmax] for the PBC model
      A*exp(-r/xi)/sqrt(r) + periodic image
    with NO constant offset.

    Inputs:
      C: (L,) correlator at r=0..L-1
      C_cov: (L,L) covariance matrix for C
      L: lattice size
      rmin: minimum r included (must be >=1)
      rmax_stop: maximum rmax to try (default L//2)
      jitter: optional diagonal regularization added to each sub-covariance

    Returns:
      rmax_list, xi_list, xi_err_list, A_list, A_err_list, fit_ok
    """
    C = np.asarray(C, dtype=float)
    C_cov = np.asarray(C_cov, dtype=float)

    if rmin < 1:
        raise ValueError("rmin must be >= 1 for the 1/sqrt(r) model (avoid r=0).")

    if rmax_stop is None:
        rmax_stop = L // 2  # typical for PBC-symmetric correlators

    # Need at least 2 points to fit (A, xi)
    rmax_list = np.arange(rmin + 1, rmax_stop + 1)

    xi_list = np.full_like(rmax_list, np.nan, dtype=float)
    xi_err_list = np.full_like(rmax_list, np.nan, dtype=float)
    A_list = np.full_like(rmax_list, np.nan, dtype=float)
    A_err_list = np.full_like(rmax_list, np.nan, dtype=float)
    fit_ok = np.zeros_like(rmax_list, dtype=bool)

    for idx, rmax in enumerate(rmax_list):
        r = np.arange(rmin, rmax + 1, dtype=float)
        y = C[rmin : rmax + 1]

        cov = C_cov[rmin : rmax + 1, rmin : rmax + 1].copy()
        if jitter != 0.0:
            cov.flat[:: cov.shape[0] + 1] += jitter

        # Initial guesses:
        # xi: some fraction of L
        xi0 = max(1.0, 0.15 * L)
        # A from the first point: y(rmin) ~ A * exp(-rmin/xi)/sqrt(rmin) (ignore image term for init)
        A0 = float(y[0] * np.sqrt(rmin) * np.exp(rmin / xi0))

        p0 = (A0, xi0)
        bounds = ([-np.inf, 1e-6], [np.inf, 10.0 * L])

        try:
            popt, pcov = curve_fit(
                lambda rr, A, xi: corr_pbc_exp_over_sqrt(rr, A, xi, L),
                r,
                y,
                p0=p0,
                sigma=cov,              # full covariance => correlated fit
                absolute_sigma=True,
                bounds=bounds,
                maxfev=20000,
            )
            A_hat, xi_hat = popt
            A_list[idx] = A_hat
            xi_list[idx] = xi_hat

            # parameter errors from fit covariance
            A_err_list[idx] = np.sqrt(max(pcov[0, 0], 0.0))
            xi_err_list[idx] = np.sqrt(max(pcov[1, 1], 0.0))
            fit_ok[idx] = True
        except Exception:
            fit_ok[idx] = False

    return rmax_list, xi_list, xi_err_list, A_list, A_err_list, fit_ok


def plot_xi_vs_window(rmax_list, xi_list, xi_err_list, fit_ok, title=None):
    mask = fit_ok & np.isfinite(xi_list) & np.isfinite(xi_err_list)
    plt.figure()
    plt.errorbar(rmax_list[mask], xi_list[mask], yerr=xi_err_list[mask], fmt="o", capsize=3)
    plt.xlabel("Fit window max separation r_max")
    plt.ylabel("Fitted correlation length xi")
    if title is not None:
        plt.title(title)
    plt.grid(True)
    plt.show()

In [2]:
def xi_second_moment_connected_block_jackknife(samples, block_size, show_progress=True):
    """
    Compute the second-moment correlation length xi_2 with a block-jackknife
    error estimate, using precomputed block correlators and block field sums.

    The connected, translation-averaged correlator with PBCs is
        C_conn(r) = C(r) - D(r)

    where
        C(r) = < (1/L) sum_x n_x · n_{x+r} >
        D(r) = (1/L) sum_x m_x · m_{x+r}
        m_x  = < n_x >

    The second-moment xi is then
        xi^2 = [1 / (4 sin^2(k_min/2))] * (G(0)/G(k_min) - 1),
        k_min = 2 pi / L,
        G(k) = sum_r cos(k r) C_conn(r).

    Parameters
    ----------
    samples : array, shape (Nsamp, L, 3)
    block_size : int
        Contiguous block size for jackknife.
    show_progress : bool
        Whether to show a tqdm bar over jackknife blocks.

    Returns
    -------
    xi_gvar : gvar.gvar
        Jackknife estimate of xi_2.
    C_conn_full : np.ndarray, shape (L,)
        Full-sample connected correlator used for the central value.
    """
    samples = np.asarray(samples, dtype=float)
    N, L, d = samples.shape
    if d != 3:
        raise ValueError(f"Expected samples shape (Nsamp, L, 3), got {samples.shape}")
    if block_size < 1:
        raise ValueError("block_size must be >= 1")

    Nb = N // block_size
    if Nb < 2:
        raise ValueError("Need at least 2 full blocks for jackknife.")

    Ntrim = Nb * block_size
    samples = samples[:Ntrim]

    # ------------------------------------------------------------
    # Per-sample translation-averaged correlators using FFT:
    # c_s(r) = (1/L) sum_x n_s[x] · n_s[x+r]
    # ------------------------------------------------------------
    fk = np.fft.rfft(samples, axis=1)                          # (Ntrim, Lf, 3)
    power = np.sum(np.abs(fk) ** 2, axis=-1)                  # (Ntrim, Lf)
    corr_per_sample = np.fft.irfft(power, n=L, axis=1) / L    # (Ntrim, L)

    # Block sums of correlators and block sums of fields
    corr_blocks = corr_per_sample.reshape(Nb, block_size, L)
    corr_sum_blocks = np.sum(corr_blocks, axis=1)             # (Nb, L)

    field_blocks = samples.reshape(Nb, block_size, L, 3)
    field_sum_blocks = np.sum(field_blocks, axis=1)           # (Nb, L, 3)

    corr_sum_total = np.sum(corr_sum_blocks, axis=0)          # (L,)
    field_sum_total = np.sum(field_sum_blocks, axis=0)        # (L, 3)

    # ------------------------------------------------------------
    # Helper: connected correlator and xi from sums over samples
    # ------------------------------------------------------------
    def connected_corr_from_sums(corr_sum, field_sum, count):
        C = corr_sum / count                                  # (L,)
        m = field_sum / count                                 # (L, 3)
        D = np.mean(np.sum(m * np.roll(m, shift=-np.arange(L)[:, None], axis=0), axis=-1), axis=1)
        # The line above is awkward; do it clearly below instead.

    def connected_corr_from_means(C_mean, m_mean):
        # D(r) = (1/L) sum_x m_x · m_{x+r}
        D = np.empty(L, dtype=float)
        for r in range(L):
            D[r] = np.mean(np.sum(m_mean * np.roll(m_mean, shift=-r, axis=0), axis=-1))
        return C_mean - D

    def xi_from_corr(C_conn):
        k_min = 2.0 * np.pi / L
        r = np.arange(L, dtype=float)
        G0 = np.sum(C_conn)
        Gk = np.sum(np.cos(k_min * r) * C_conn)
        xi2 = (G0 / Gk - 1.0) / (4.0 * np.sin(k_min / 2.0) ** 2)
        return np.sqrt(max(xi2, 0.0))

    # Full-sample central value
    C_full = corr_sum_total / Ntrim
    m_full = field_sum_total / Ntrim
    C_conn_full = connected_corr_from_means(C_full, m_full)
    xi_full = xi_from_corr(C_conn_full)

    # ------------------------------------------------------------
    # Jackknife leave-one-block-out estimates
    # ------------------------------------------------------------
    xi_loo = np.empty(Nb, dtype=float)

    iterator = range(Nb)
    if show_progress:
        iterator = tqdm.tqdm(iterator, total=Nb, desc="Block jackknife xi_2", leave=True)

    count_loo = Ntrim - block_size
    for b in iterator:
        C_loo = (corr_sum_total - corr_sum_blocks[b]) / count_loo
        m_loo = (field_sum_total - field_sum_blocks[b]) / count_loo
        C_conn_loo = connected_corr_from_means(C_loo, m_loo)
        xi_loo[b] = xi_from_corr(C_conn_loo)

    xi_bar = np.mean(xi_loo)
    xi_var = (Nb - 1) / Nb * np.sum((xi_loo - xi_bar) ** 2)
    xi_err = np.sqrt(max(xi_var, 0.0))

    return gvar.gvar(xi_full, xi_err), C_conn_full

In [ ]:
# second moment
xi, xi_err = xi_second_moment_from_corr(C, C_cov)
gvar.gvar(xi,xi_err)

In [ ]:
# bessel
L = len(C)
rmax_list, xi_list, xi_err_list, A_list, A_err_list, fit_ok = fit_xi_windows_correlated(
    C, C_cov, L, rmin=2, rmax_stop=L//2, jitter=1e-12 * np.trace(C_cov) / C_cov.shape[0]
)
plot_xi_vs_window(rmax_list, xi_list, xi_err_list, fit_ok, title="xi vs fit window")


### Old

In [ ]:
N = 80
eta = 1
gsq = .73
g = np.sqrt(gsq) # beta = 1/g^2
mu = 0
N_CORES = 16

x = np.random.normal(size=(N, 3))
model = MLP_SO3(hidden_sizes=(N,))

# strip the leading 0 from the string representation of gsq
gsq_str = str(gsq).lstrip("0")
params = np.load(f"data/L_40_g2_{gsq_str}_params.npy", allow_pickle=True).item()
y = model.apply(params,x)

model_apply = jit(model.apply)

psi = model_apply

def gen_init_positions(nchains, input):
    pos_initials = [jax.random.normal(jax.random.PRNGKey(i + int(input)), shape=(N, 3)) for i in range(nchains)]
    # for each chain, for each site, normalize the 3D vector to lie on the unit sphere
    for i, x in enumerate(pos_initials):
        pos_initials[i] /= jnp.linalg.norm(x, axis=-1, keepdims=True)
    return pos_initials


sampler = newSampler(psi, (N, 3))

nchains = N_CORES
pos_initials = gen_init_positions(nchains, int(time.time()))
seeds = jnp.arange(nchains)
var = 0.35

samples, acc_rate = sampler.run_many_chains(params, nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")


Acceptance rate: 0.5041720271110535


In [ ]:
samples, acc_rate = sampler.run_many_chains(params, 1000000//16, 50000, 10, var, gen_init_positions(16, int(time.time())), [int(time.time()) + i for i in range(16)])
print(f"Acceptance rate: {acc_rate}")


C, cov, C_uncerts = Cr_with_cov_optimized(samples, batch_size=64)


ui = fitting_widget(
    C,
    cov,
    initial_fit_type="exponential",
    initial_rel_cut=1e-5,
)

In [ ]:
# After clicking "Extract plateau ξ" in the UI:
extracted_xi = ui["result_state"]["xi"]

# print(extracted_xi.mean, extracted_xi.sdev)
print(f"Extracted ξ (g^2 = {gsq}): {extracted_xi}")

In [ ]:
gse, gseu = batched_energy(model, eta, g, mu, params, samples, batch_size=16)
print(f"GSE: {gse} ± {gseu}")